# 03 - Baseline Revenue Backcast

This notebook calculates historical revenue for a renewable asset:
- Configure asset parameters (capacity, type)
- Calculate merchant revenue: Revenue = Price × Generation
- Analyze capacity factor and capture price
- Visualize revenue patterns and seasonality
- Compare different revenue contract structures

## Outputs
- Revenue backcast results
- Revenue visualizations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src to path
sys.path.append('../')

from src.models import RevenueCalculator, RevenueScenario, backcast_historical_revenue
from src.utils import load_config, plot_monthly_revenue, create_dashboard

# Set display options
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Data and Configuration

In [ ]:
# Load configuration
config = load_config('../configs/modelling_config.yaml')

# Load clean data
data_path = Path(config.get('data.processed_path', '../data_processed'))
df = pd.read_csv(data_path / 'clean_data.csv', index_col=0, parse_dates=True)

print(f"Loaded data: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")

# Asset configuration
asset_capacity_mw = config.get('asset.capacity_mw', 100)
asset_type = config.get('asset.type', 'offshore_wind')

print(f"\nAsset: {asset_capacity_mw}MW {asset_type}")

## 2. Calculate Baseline Merchant Revenue

In [ ]:
# Initialize revenue scenario
scenario = RevenueScenario(
    asset_capacity_mw=asset_capacity_mw,
    asset_type=asset_type
)

# Calculate merchant revenue
result = scenario.run_scenario(df, scenario_type='merchant')

# Display metrics
metrics = result.attrs['metrics']
print("\n" + "="*60)
print("MERCHANT REVENUE METRICS")
print("="*60)
for key, value in metrics.items():
    if 'revenue' in key or 'price' in key:
        print(f"{key.replace('_', ' ').title()}: £{value:,.0f}")
    else:
        print(f"{key.replace('_', ' ').title()}: {value:.4f}")

## 3. Revenue Visualization

In [ ]:
# Plot revenue time series
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Price
result['price'].plot(ax=axes[0], linewidth=0.5, alpha=0.7)
axes[0].set_ylabel('Price (£/MWh)')
axes[0].set_title('Electricity Price')
axes[0].grid(True, alpha=0.3)

# Generation
result[asset_type].plot(ax=axes[1], linewidth=0.5, alpha=0.7)
axes[1].set_ylabel('Generation (MWh)')
axes[1].set_title(f'{asset_type.replace("_", " ").title()} Generation')
axes[1].grid(True, alpha=0.3)

# Revenue
result['revenue'].plot(ax=axes[2], linewidth=0.5, alpha=0.7)
axes[2].set_ylabel('Revenue (£)')
axes[2].set_title('Hourly Revenue')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Monthly revenue plot
monthly_revenue = result['revenue'].resample('M').sum()

fig, ax = plt.subplots(figsize=(14, 6))
monthly_revenue.plot(kind='bar', ax=ax, alpha=0.7, edgecolor='black')
ax.set_ylabel('Revenue (£)')
ax.set_title('Monthly Revenue')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"\nTotal annual revenue: £{result['revenue'].sum():,.0f}")
print(f"Average monthly revenue: £{monthly_revenue.mean():,.0f}")

## 4. Seasonal Analysis

In [ ]:
# Add time features for analysis
result['month'] = result.index.month
result['hour'] = result.index.hour
result['day_of_week'] = result.index.dayofweek

# Revenue by month
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Revenue by month
monthly_avg = result.groupby('month')['revenue'].sum()
monthly_avg.plot(kind='bar', ax=axes[0, 0], alpha=0.7)
axes[0, 0].set_xlabel('Month')
axes[0, 0].set_ylabel('Total Revenue (£)')
axes[0, 0].set_title('Revenue by Month')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Revenue by hour
hourly_avg = result.groupby('hour')['revenue'].mean()
hourly_avg.plot(kind='bar', ax=axes[0, 1], alpha=0.7)
axes[0, 1].set_xlabel('Hour')
axes[0, 1].set_ylabel('Average Revenue (£)')
axes[0, 1].set_title('Average Revenue by Hour')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Generation by month
monthly_gen = result.groupby('month')[asset_type].sum()
monthly_gen.plot(kind='bar', ax=axes[1, 0], alpha=0.7)
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Total Generation (MWh)')
axes[1, 0].set_title('Generation by Month')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Price by month
monthly_price = result.groupby('month')['price'].mean()
monthly_price.plot(kind='bar', ax=axes[1, 1], alpha=0.7)
axes[1, 1].set_xlabel('Month')
axes[1, 1].set_ylabel('Average Price (£/MWh)')
axes[1, 1].set_title('Average Price by Month')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Compare Revenue Scenarios

In [ ]:
# Define scenarios
ppa_price = config.get('scenarios.ppa_price', 50)
cfd_strike_price = config.get('scenarios.cfd_strike_price', 55)
ppa_fraction = config.get('scenarios.ppa_fraction', 0.5)

scenarios = {
    'merchant': {'scenario_type': 'merchant'},
    'ppa_50': {'scenario_type': 'ppa', 'ppa_price': ppa_price},
    'cfd_55': {'scenario_type': 'cfd', 'strike_price': cfd_strike_price},
    'blended_50_50': {'scenario_type': 'blended', 'ppa_price': ppa_price, 'ppa_fraction': ppa_fraction}
}

# Run scenarios
comparison = scenario.compare_scenarios(df, scenarios)
print("\nScenario Comparison:")
print(comparison)

In [ ]:
# Visualize scenario comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total revenue comparison
comparison['total_revenue'].plot(kind='bar', ax=axes[0], alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Total Revenue (£)')
axes[0].set_title('Total Revenue by Scenario')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].tick_params(axis='x', rotation=45)

# Add value labels
for i, v in enumerate(comparison['total_revenue']):
    axes[0].text(i, v, f'£{v:,.0f}', ha='center', va='bottom')

# Capture price comparison
comparison['capture_price'].plot(kind='bar', ax=axes[1], alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Capture Price (£/MWh)')
axes[1].set_title('Capture Price by Scenario')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].tick_params(axis='x', rotation=45)

# Add value labels
for i, v in enumerate(comparison['capture_price']):
    axes[1].text(i, v, f'£{v:.2f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 6. Capacity Factor Analysis

In [ ]:
# Calculate monthly capacity factors
monthly_gen = result[asset_type].resample('M').sum()
days_per_month = result[asset_type].resample('M').count() / 24  # Hours to days
monthly_cf = monthly_gen / (asset_capacity_mw * days_per_month * 24)

fig, ax = plt.subplots(figsize=(12, 5))
monthly_cf.plot(kind='bar', ax=ax, alpha=0.7, edgecolor='black')
ax.set_ylabel('Capacity Factor')
ax.set_title('Monthly Capacity Factor')
ax.axhline(y=monthly_cf.mean(), color='r', linestyle='--', label=f'Average: {monthly_cf.mean():.2%}')
ax.grid(True, alpha=0.3, axis='y')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"Average capacity factor: {metrics['capacity_factor']:.2%}")
print(f"Min monthly CF: {monthly_cf.min():.2%}")
print(f"Max monthly CF: {monthly_cf.max():.2%}")

## 7. Save Results

In [ ]:
# Save revenue results
output_path = Path(config.get('data.processed_path', '../data_processed'))
output_file = output_path / 'revenue_backcast.csv'

result[['price', asset_type, 'revenue']].to_csv(output_file)

print(f"\n✓ Revenue backcast saved to: {output_file}")

# Save comparison
comparison_file = output_path / 'scenario_comparison.csv'
comparison.to_csv(comparison_file)
print(f"✓ Scenario comparison saved to: {comparison_file}")

## Summary

Baseline revenue backcast complete!

**Key findings:**
- Historical revenue patterns analyzed
- Capacity factor calculated
- Capture price vs market price computed
- Multiple contract structures compared

**Next steps:**
- Proceed to notebook 04 for price modelling
- Train ML models to forecast prices
- Evaluate model performance